In [1]:
from git import Repo
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
# Model and Embeddings integrations are now in langchain_openai
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_openai import ChatOpenAI
# Chroma has its own dedicated community package now
from langchain_chroma import Chroma
# Memory and Chains components remain in the core langchain package
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationalRetrievalChain
import os




c:\saifulla office\Data science and genrative AI krish naik\Projects\Realtime-Source-Code-Analyser\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
!mkdir -p test_repo

In [6]:
repo_path = 'test_repo'
repo = Repo.clone_from("https://github.com/soudagar/AI-Book-Management-System.git",to_path=repo_path)


GitCommandError: Cmd('git') failed due to: exit code(128)
  cmdline: git clone -v -- https://github.com/soudagar/AI-Book-Management-System.git test_repo
  stderr: 'fatal: destination path 'test_repo' already exists and is not an empty directory.
'

In [8]:
loader = GenericLoader.from_filesystem(repo_path,
                                       glob="**/*",
                                       suffixes=[".py"],
                                       parser = LanguageParser(language="python", parser_threshold=500)
                                       )

In [9]:
document_loader = loader.load()

In [10]:
len(document_loader)

39

In [12]:
document_loader[0]

Document(metadata={'source': 'test_repo\\backend\\fix_db.py', 'language': 'python'}, page_content='import asyncio\nfrom sqlalchemy import text\nfrom app.core.database import engine\n\nasync def fix_schema():\n    print("Connecting to database...")\n    async with engine.begin() as conn:\n        print("Checking \'books\' table schema...")\n        try:\n            # Attempt to add \'rolling_consensus\' column if it doesn\'t exist\n            # Note: IF NOT EXISTS is PostgreSQL syntax.\n            await conn.execute(text("ALTER TABLE books ADD COLUMN IF NOT EXISTS rolling_consensus TEXT;"))\n            print("Successfully added/verified \'rolling_consensus\' column.")\n            \n            # Also check average_rating just in case\n            await conn.execute(text("ALTER TABLE books ADD COLUMN IF NOT EXISTS average_rating FLOAT DEFAULT 0.0;"))\n            print("Successfully added/verified \'average_rating\' column.")\n\n        except Exception as e:\n            print(f"Er

In [11]:
document_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=500,
    chunk_overlap=20
)

In [12]:
texts = document_splitter.split_documents(document_loader)

In [15]:
len(texts)

326

In [16]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")


In [17]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [13]:
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11612.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
vector_db = Chroma.from_documents(texts, embedding=embedding, persist_directory='./data' )

In [23]:
llm = ChatOllama(model="llama3.2:latest")

In [24]:
memory = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)


In [25]:
qa = ConversationalRetrievalChain.from_llm(llm, retriever=vector_db.as_retriever(search_type="mmr", search_kwargs={"k":8}),memory=memory)


In [18]:
question = "explain create_book function from code "

In [26]:
result = qa(question)

In [28]:
print(result['answer'])

The `create_book` function is a static method in the `BookService` class. It's used to create a new book entry in the database.

Here's a breakdown of what it does:

1. **Arguments**:
   - `db`: The database session object, which provides access to the database.
   - `owner_id`: The ID of the user who owns the book.
   - `book_data`: A dictionary containing the data for the new book.

2. **Creation Process**:
   - It creates a new instance of the `Book` class using the provided data and owner ID.
   - The `**book_data.dict()` part is used to unpack the `book_data` dictionary into keyword arguments, which are then passed to the `Book` constructor.
   - The created book is then added to the database session (`db.add(book)`).
   - The changes are committed to the database (`await db.commit()`).
   - Finally, the book instance is refreshed from the database to include any changes made during the commit process (`await db.refresh(book)`).

3. **Error Handling**:
   - If an exception occurs 

In [29]:
question = "explain the summary of this repo"

In [30]:
result = qa(question)

In [31]:
print(result["answer"])

This repository appears to be a Python FastAPI application that provides a conversational AI interface using a Large Language Model (LLaMA) service. It allows users to ask questions and receive answers through an API endpoint, with features such as document retrieval and book recommendation routes also available. The code snippets suggest that it handles tasks like text embedding, similarity calculation, and database interactions, but the exact functionality and details of the application's architecture are not explicitly stated in the provided context.
